# LeetCode 4113 — Game of Divisible Sums (Medium)

Bài toán LeetCode:

```python
class Solution:
    def maxDifference(self, nums: List[int]) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4113 (Weekly Contest 528 — câu 3).

Đây là bài **trò chơi hai người chơi tối ưu** — một dạng quy hoạch động rất hay gặp. Notebook đi từ con số 0:

1. "Chơi tối ưu" nghĩa là gì, vì sao "tối đa điểm của mình" tương đương "tối đa **hiệu** điểm".
2. **Minimax**: Alice tối đa, Bob tối thiểu — đúng nhưng bùng nổ.
3. **Negamax / DP theo hiệu điểm**: trạng thái chỉ còn là "vị trí bắt đầu `i`", và luật "được đi tiếp" chỉ là **đổi dấu**.
4. Ghi nhớ (memoization) → DP từ dưới lên → tối ưu bộ nhớ `O(1)`.

---

## Sau notebook này, bạn sẽ nắm được

1. Đề bài và cách đọc luật "tổng chia hết cho 3 thì đổi lượt".
2. Cây trò chơi (game tree), đệ quy minimax, và vì sao số lời gọi tăng theo hàm mũ.
3. Mẹo **hiệu điểm** (negamax): `dfs(i)` = chênh lệch tốt nhất mà **người đang đi** đạt được từ hậu tố `i`.
4. `functools.lru_cache`, giới hạn đệ quy của Python, và cách chuyển sang DP lặp.
5. Lỗi hay gặp (luôn đổi lượt, sai dấu, tham lam...), mẹo nhận diện họ bài *Stone Game*, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Alice và Bob chơi với mảng số nguyên dương `nums`. **Alice đi trước.**

Ở mỗi lượt, người chơi hiện tại lấy ra:

- **phần tử ngoài cùng bên trái**, hoặc
- **hai phần tử ngoài cùng bên trái** (nếu còn ít nhất 2 phần tử).

Gọi `s` là tổng các phần tử vừa lấy trong lượt đó. Luật quyết định ai đi tiếp:

| Tổng `s` của lượt vừa đi | Ai đi lượt kế tiếp |
|---|---|
| `s` **chia hết** cho 3 | **người kia** |
| `s` **không chia hết** cho 3 | **cùng người đó** đi tiếp |

Điểm của mỗi người là tổng các phần tử họ đã lấy. Trò chơi kết thúc khi mảng rỗng. Cả hai đều **chơi tối ưu** để tối đa điểm của chính mình.

Trả về **điểm của Alice trừ điểm của Bob**.

---

## Ví dụ 1

```python
nums = [3, 6, 4]   ->   5
```

- Alice lấy hai phần tử đầu: `3 + 6 = 9`, chia hết cho 3 → Bob đi tiếp. Alice có `9`.
- Bob lấy `4`, hết mảng.
- Kết quả `9 - 4 = 5`.

## Ví dụ 2

```python
nums = [6, 1, 2, 1, 2]   ->   12
```

- Alice lấy `6 + 1 = 7` (không chia hết) → Alice đi tiếp. Alice có `7`.
- Alice lấy `2` (không chia hết) → đi tiếp. Alice có `9`.
- Alice lấy `1 + 2 = 3` → hết mảng. Alice có `12`, Bob có `0`.
- Kết quả `12`.

## Ví dụ 3

```python
nums = [3, 3, 4, 5]   ->   -3
```

- Alice lấy `3 + 3 = 6` (chia hết) → Bob đi tiếp. Alice có `6`.
- Bob lấy `4 + 5 = 9`, hết mảng.
- Kết quả `6 - 9 = -3`. (Alice không có cách nào tốt hơn — ta sẽ kiểm chứng.)

---

## Ràng buộc

```text
1 <= nums.length <= 10^5
1 <= nums[i] <= 10^9
```

# 2. Kiến thức nền tảng

## 2.1. "Chơi tối ưu" và mẹo hiệu điểm

Mọi phần tử cuối cùng đều thuộc về **một trong hai** người. Nếu tổng mảng là `T`:

```text
điểm Alice + điểm Bob = T     (cố định!)
```

Do đó, với người đang đi:

```text
tối đa điểm của mình   ⇔   tối đa (điểm mình - điểm đối thủ)
```

vì `mình - đối thủ = 2 · mình - T`. Mục tiêu của mỗi người trở thành **tối đa hiệu điểm theo góc nhìn của chính họ**. Và đề hỏi đúng hiệu điểm `Alice - Bob`, nên ta làm việc trực tiếp với hiệu.

Cell dưới minh họa: các phương án chia điểm khác nhau có cùng tổng, nên xếp hạng theo "điểm mình" hay theo "hiệu" là như nhau.

In [1]:
T = 13                                  # tổng của [3, 6, 4]
for alice in [3, 9, 13]:
    bob = T - alice
    print(f"Alice = {alice:>2}, Bob = {bob:>2} -> hiệu = {alice - bob:>3} = 2*{alice} - {T}")

Alice =  3, Bob = 10 -> hiệu =  -7 = 2*3 - 13
Alice =  9, Bob =  4 -> hiệu =   5 = 2*9 - 13
Alice = 13, Bob =  0 -> hiệu =  13 = 2*13 - 13


## 2.2. Cây trò chơi (game tree)

Từ một trạng thái, mỗi nước đi dẫn tới một trạng thái mới. Vẽ ra ta được một **cây**. Với `[3, 6, 4]`:

```text
                     [3,6,4]  Alice đi
                 /                      \
     lấy 3 (chia hết → Bob)        lấy 3+6=9 (chia hết → Bob)
           [6,4]  Bob đi                  [4]  Bob đi
         /            \                      |
 lấy 6 (→ Alice)   lấy 6+4=10 (Bob tiếp)   lấy 4 → hết
     [4] Alice          hết                 hiệu = 9 - 4 = 5
       |              hiệu = 3 - 10 = -7
   lấy 4 → hết
   hiệu = 3 + 4 - 6 = 1
```

Bob ở nhánh trái sẽ chọn nước làm hiệu **nhỏ nhất** (tốt cho Bob): `min(1, -7) = -7`. Alice ở gốc chọn **lớn nhất**: `max(-7, 5) = 5`. Đáp án `5` ✔.

## 2.3. Ghi nhớ kết quả bằng `functools.lru_cache`

Gắn `@lru_cache(maxsize=None)` lên một hàm thuần (cùng tham số → cùng kết quả), Python sẽ **nhớ** kết quả của mỗi bộ tham số và không tính lại. Ví dụ kinh điển — Fibonacci:

In [2]:
from functools import lru_cache

calls = 0
def fib_plain(n):
    global calls
    calls += 1
    return n if n < 2 else fib_plain(n - 1) + fib_plain(n - 2)

fib_plain(25)
print("Không nhớ   : fib(25) gọi hàm", f"{calls:,}", "lần")

calls = 0
@lru_cache(maxsize=None)
def fib_memo(n):
    global calls
    calls += 1
    return n if n < 2 else fib_memo(n - 1) + fib_memo(n - 2)

fib_memo(25)
print("Có lru_cache: fib(25) gọi hàm", calls, "lần")

Không nhớ   : fib(25) gọi hàm 242,785 lần
Có lru_cache: fib(25) gọi hàm 26 lần


## 2.4. Giới hạn đệ quy

Python mặc định chỉ cho đệ quy sâu khoảng 1000 tầng (`sys.getrecursionlimit()`). Bài này có thể cần đệ quy sâu `10^5` tầng → phải chuyển sang **DP lặp** khi nộp bài.

In [3]:
import sys
print("Giới hạn đệ quy hiện tại:", sys.getrecursionlimit())

Giới hạn đệ quy hiện tại: 1000


# 3. Cách 1 — Minimax: lưu cả "đến lượt ai"

Trạng thái gồm: vị trí `i` (phần còn lại là `nums[i:]`) và **ai đang đi**. Hàm trả về **hiệu `Alice - Bob`** của phần còn lại khi cả hai chơi tối ưu:

- Alice đi → chọn nước cho hiệu **lớn nhất**; điểm Alice lấy được **cộng** vào hiệu.
- Bob đi → chọn nước cho hiệu **nhỏ nhất**; điểm Bob lấy được **trừ** khỏi hiệu.
- Sau mỗi nước: tổng chia hết cho 3 → đổi người; ngược lại → giữ nguyên người.

In [4]:
from typing import List


def max_diff_minimax(nums: List[int]) -> int:
    n = len(nums)

    def go(i: int, alice_turn: bool) -> int:
        if i == n:
            return 0
        results = []
        for take in (1, 2):
            if i + take > n:
                break
            s = sum(nums[i:i + take])
            next_alice = (not alice_turn) if s % 3 == 0 else alice_turn
            rest = go(i + take, next_alice)
            results.append(rest + s if alice_turn else rest - s)
        return max(results) if alice_turn else min(results)

    return go(0, True)


print(max_diff_minimax([3, 6, 4]))         # 5
print(max_diff_minimax([6, 1, 2, 1, 2]))   # 12
print(max_diff_minimax([3, 3, 4, 5]))      # -3

5
12
-3


## Số lời gọi tăng theo hàm mũ

Mỗi trạng thái có tới 2 nước đi → số lời gọi tăng giống dãy Fibonacci (khoảng `1.6^n`).

In [5]:
def count_minimax_calls(nums: List[int]) -> int:
    n = len(nums)
    counter = [0]

    def go(i, alice_turn):
        counter[0] += 1
        if i == n:
            return 0
        res = []
        for take in (1, 2):
            if i + take > n:
                break
            s = sum(nums[i:i + take])
            nxt = (not alice_turn) if s % 3 == 0 else alice_turn
            r = go(i + take, nxt)
            res.append(r + s if alice_turn else r - s)
        return max(res) if alice_turn else min(res)

    go(0, True)
    return counter[0]


for n_ in [10, 15, 20, 25]:
    print(f"n = {n_}: {count_minimax_calls([1] * n_):,} lời gọi")

n = 10: 232 lời gọi
n = 15: 2,583 lời gọi
n = 20: 28,656 lời gọi
n = 25: 317,810 lời gọi


Với `n = 10^5` là không tưởng. Có thể thêm `lru_cache` với trạng thái `(i, alice_turn)` để còn `2n` trạng thái — nhưng còn một cách nhìn gọn hơn nhiều.

---

# 4. Bước đột phá: bỏ "đến lượt ai" khỏi trạng thái (negamax)

Quan sát: khi đứng trước hậu tố `nums[i:]`, **bài toán của Alice và của Bob y hệt nhau** — cả hai đều muốn tối đa "điểm mình − điểm đối thủ" trên phần còn lại. Vậy chỉ cần một hàm, tính **theo góc nhìn của người đang đi**:

> `dfs(i)` = hiệu điểm (người đang đi − người kia) lớn nhất có thể đạt được trên hậu tố `nums[i:]`.

Người đang đi lấy một nhóm có tổng `s`, rồi:

- **`s` chia hết cho 3 → đổi lượt.** Phần còn lại do **đối thủ** đi trước; theo góc nhìn đối thủ hiệu là `dfs(next)`, nên theo góc nhìn của mình là `-dfs(next)`. Tổng: `s - dfs(next)`.
- **`s` không chia hết → mình đi tiếp.** Phần còn lại vẫn do **mình** đi trước: `s + dfs(next)`.

```text
dfs(n) = 0
dfs(i) = max(  x  ± dfs(i + 1),          với x = nums[i]
             (x + y) ± dfs(i + 2) )      với y = nums[i + 1], nếu i + 1 < n
dấu "-" nếu tổng chia hết cho 3 (đổi lượt), dấu "+" nếu không (đi tiếp)
answer = dfs(0)
```

Luật "ai đi tiếp" biến thành **một dấu cộng / trừ** — gọn hơn hẳn minimax.

## Tính tay với ví dụ 3: `nums = [3, 3, 4, 5]`

Tính từ cuối lên (vì `dfs(i)` cần `dfs(i + 1)`, `dfs(i + 2)`):

| i | lấy 1: `x`, chia hết? | giá trị | lấy 2: `x + y`, chia hết? | giá trị | dfs(i) |
|---:|---|---:|---|---:|---:|
| 4 | — | | — | | 0 |
| 3 | `5`, không | `5 + 0 = 5` | — | | 5 |
| 2 | `4`, không | `4 + 5 = 9` | `9`, có | `9 - 0 = 9` | 9 |
| 1 | `3`, có | `3 - 9 = -6` | `7`, không | `7 + 5 = 12` | 12 |
| 0 | `3`, có | `3 - 12 = -9` | `6`, có | `6 - 9 = -3` | **-3** |

`dfs(0) = -3` ✔. Cả hai nước đầu của Alice đều chuyển lượt cho Bob ở vị trí rất có lợi cho Bob; lấy `3 + 3` là "đỡ tệ" nhất.

# 5. Cách 2 — Negamax + ghi nhớ: O(n)

In [6]:
def max_diff_memo(nums: List[int]) -> int:
    n = len(nums)

    @lru_cache(maxsize=None)
    def dfs(i: int) -> int:
        if i == n:
            return 0
        x = nums[i]
        best = x - dfs(i + 1) if x % 3 == 0 else x + dfs(i + 1)
        if i + 1 < n:
            s = x + nums[i + 1]
            best = max(best, s - dfs(i + 2) if s % 3 == 0 else s + dfs(i + 2))
        return best

    return dfs(0)


print(max_diff_memo([3, 6, 4]))         # 5
print(max_diff_memo([6, 1, 2, 1, 2]))   # 12
print(max_diff_memo([3, 3, 4, 5]))      # -3

5
12
-3


Chỉ có `n + 1` trạng thái `i = 0..n`, mỗi trạng thái tính một lần trong `O(1)` → `O(n)`. Nhưng độ sâu đệ quy vẫn tới `n` → với `n = 10^5` sẽ gặp `RecursionError` (xem mục 10).

---

# 6. Cách 3 — DP lặp từ cuối mảng: O(n)

Vì `dfs(i)` chỉ phụ thuộc `dfs(i + 1)` và `dfs(i + 2)`, ta tính mảng `f` **từ `i = n - 1` lùi về `0`**, không cần đệ quy. Thêm `f[n] = f[n + 1] = 0` làm lính canh.

## Bảng DP cho ví dụ 2: `nums = [6, 1, 2, 1, 2]`

| i | nums[i] | lấy 1 | lấy 2 | f[i] |
|---:|---:|---|---|---:|
| 5 | — | | | 0 |
| 4 | 2 | `2` không chia hết → `2 + 0 = 2` | — | 2 |
| 3 | 1 | `1` → `1 + 2 = 3` | `1+2 = 3` chia hết → `3 - 0 = 3` | 3 |
| 2 | 2 | `2` → `2 + 3 = 5` | `2+1 = 3` chia hết → `3 - 2 = 1` | 5 |
| 1 | 1 | `1` → `1 + 5 = 6` | `1+2 = 3` chia hết → `3 - 3 = 0` | 6 |
| 0 | 6 | `6` chia hết → `6 - 6 = 0` | `6+1 = 7` → `7 + 5 = 12` | **12** |

`f[0] = 12` ✔ — Alice tránh mọi nước làm tổng chia hết cho 3 để giữ lượt tới cuối.

In [7]:
def max_diff_bottom_up(nums: List[int]) -> int:
    n = len(nums)
    f = [0] * (n + 2)                        # f[n] = f[n+1] = 0
    for i in range(n - 1, -1, -1):
        x = nums[i]
        f[i] = x - f[i + 1] if x % 3 == 0 else x + f[i + 1]
        if i + 1 < n:
            s = x + nums[i + 1]
            f[i] = max(f[i], s - f[i + 2] if s % 3 == 0 else s + f[i + 2])
    return f[0]


print(max_diff_bottom_up([3, 6, 4]))         # 5
print(max_diff_bottom_up([6, 1, 2, 1, 2]))   # 12
print(max_diff_bottom_up([3, 3, 4, 5]))      # -3

5
12
-3


In [8]:
def trace_bottom_up(nums: List[int]) -> int:
    n = len(nums)
    f = [0] * (n + 2)
    print(f"nums = {nums}")
    for i in range(n - 1, -1, -1):
        x = nums[i]
        one = x - f[i + 1] if x % 3 == 0 else x + f[i + 1]
        desc = f"lấy {x}" + (f" (chia hết → đổi lượt): {x} - {f[i + 1]} = {one}" if x % 3 == 0
                            else f" (đi tiếp): {x} + {f[i + 1]} = {one}")
        f[i] = one
        if i + 1 < n:
            s = x + nums[i + 1]
            two = s - f[i + 2] if s % 3 == 0 else s + f[i + 2]
            desc += f" | lấy {x}+{nums[i + 1]}={s}" + (f" (đổi lượt): {s} - {f[i + 2]} = {two}" if s % 3 == 0
                                                     else f" (đi tiếp): {s} + {f[i + 2]} = {two}")
            f[i] = max(one, two)
        print(f"i = {i}: {desc} -> f[{i}] = {f[i]}")
    print("Đáp án f[0] =", f[0])
    return f[0]


trace_bottom_up([6, 1, 2, 1, 2]);

nums = [6, 1, 2, 1, 2]
i = 4: lấy 2 (đi tiếp): 2 + 0 = 2 -> f[4] = 2
i = 3: lấy 1 (đi tiếp): 1 + 2 = 3 | lấy 1+2=3 (đổi lượt): 3 - 0 = 3 -> f[3] = 3
i = 2: lấy 2 (đi tiếp): 2 + 3 = 5 | lấy 2+1=3 (đổi lượt): 3 - 2 = 1 -> f[2] = 5
i = 1: lấy 1 (đi tiếp): 1 + 5 = 6 | lấy 1+2=3 (đổi lượt): 3 - 3 = 0 -> f[1] = 6
i = 0: lấy 6 (chia hết → đổi lượt): 6 - 6 = 0 | lấy 6+1=7 (đi tiếp): 7 + 5 = 12 -> f[0] = 12
Đáp án f[0] = 12


# 7. Cách 4 — Chỉ giữ hai giá trị: O(1) bộ nhớ

`f[i]` chỉ cần `f[i + 1]` và `f[i + 2]`. Giữ hai biến:

- `f1` = `f[i + 1]`,
- `f2` = `f[i + 2]`.

Sau khi tính `cur = f[i]`, dịch cửa sổ: `f2, f1 = f1, cur`.

In [9]:
def max_diff_o1(nums: List[int]) -> int:
    n = len(nums)
    f1 = f2 = 0                              # f[i+1], f[i+2]
    for i in range(n - 1, -1, -1):
        x = nums[i]
        cur = x - f1 if x % 3 == 0 else x + f1
        if i + 1 < n:
            s = x + nums[i + 1]
            cur = max(cur, s - f2 if s % 3 == 0 else s + f2)
        f2, f1 = f1, cur
    return f1                                # sau vòng lặp, f1 = f[0]


print(max_diff_o1([3, 6, 4]))         # 5
print(max_diff_o1([6, 1, 2, 1, 2]))   # 12
print(max_diff_o1([3, 3, 4, 5]))      # -3

5
12
-3


# 8. Phiên bản đúng format LeetCode

Ta nộp cách 4: không đệ quy, `O(n)` thời gian, `O(1)` bộ nhớ.

In [10]:
from typing import List


class Solution:
    def maxDifference(self, nums: List[int]) -> int:
        n = len(nums)
        f1 = f2 = 0                          # f[i+1], f[i+2]
        for i in range(n - 1, -1, -1):
            x = nums[i]
            cur = x - f1 if x % 3 == 0 else x + f1
            if i + 1 < n:
                s = x + nums[i + 1]
                cur = max(cur, s - f2 if s % 3 == 0 else s + f2)
            f2, f1 = f1, cur
        return f1

In [11]:
sol = Solution()

tests = [
    ([3, 6, 4], 5, "ví dụ 1"),
    ([6, 1, 2, 1, 2], 12, "ví dụ 2"),
    ([3, 3, 4, 5], -3, "ví dụ 3"),
    ([7], 7, "một phần tử (không chia hết)"),
    ([9], 9, "một phần tử (chia hết)"),
    ([1, 1, 1], 3, "không nước nào chia hết: Alice lấy hết"),
    ([3, 3], 6, "lấy cả hai số 3 cùng lúc"),
    ([6, 3, 1, 2], 6, "bài tập 1"),
    ([10**9, 10**9, 10**9], 3 * 10**9, "số lớn"),
]

for nums, expected, name in tests:
    result = sol.maxDifference(nums)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:42} -> {result}")

PASS | ví dụ 1                                    -> 5
PASS | ví dụ 2                                    -> 12
PASS | ví dụ 3                                    -> -3
PASS | một phần tử (không chia hết)               -> 7
PASS | một phần tử (chia hết)                     -> 9
PASS | không nước nào chia hết: Alice lấy hết     -> 3
PASS | lấy cả hai số 3 cùng lúc                   -> 6
PASS | bài tập 1                                  -> 6
PASS | số lớn                                     -> 3000000000


Giải thích test `[3, 3] -> 6`:

- Nếu Alice lấy **một** số `3` (chia hết → đổi lượt), Bob lấy số `3` còn lại → hiệu `0`.
- Nếu Alice lấy **cả hai** (`3 + 3 = 6`, chia hết → đổi lượt), nhưng mảng đã **hết** — việc trao lượt không còn tác dụng → hiệu `6`.

Bài học nhỏ: "trao lượt" chỉ có hại khi còn phần tử cho đối thủ lấy. Công thức DP tự xử lý điều này vì `f[n] = 0`.

---

# 9. Vì sao thuật toán đúng?

Chứng minh bằng quy nạp lùi theo `i` (từ `n` về `0`), với khẳng định:

> `f[i]` = hiệu điểm (người đi trước trên `nums[i:]` − người kia) khi cả hai chơi tối ưu.

- **Cơ sở:** `f[n] = 0` — không còn gì để lấy.
- **Bước quy nạp:** người đi trước tại `i` chọn lấy 1 hoặc 2 phần tử (tổng `s`), tới vị trí `j = i + 1` hoặc `i + 2`.
  - Nếu `s % 3 == 0`: người kia đi trước trên `nums[j:]` và (theo giả thiết quy nạp) đạt hiệu `f[j]` theo góc nhìn của họ, tức là `-f[j]` theo góc nhìn của mình → tổng `s - f[j]`.
  - Nếu không: mình vẫn đi trước trên `nums[j:]` → tổng `s + f[j]`.
  - Người chơi tối ưu chọn nước có giá trị lớn nhất → `f[i] = max(...)`.

Mục 2.1 đảm bảo "tối đa hiệu" và "tối đa điểm của mình" là cùng một mục tiêu, nên đây đúng là lối chơi tối ưu theo đề. Alice đi trước tại `i = 0`, nên đáp án là `f[0]`.

---

## Trực giác

> Mỗi vị trí `i` có một "giá trị" cố định: *người được đi trước ở đây sẽ hơn đối thủ bao nhiêu*. Lấy xong mà **giữ được lượt** thì cộng giá trị phía sau; **trao lượt** cho đối thủ thì trừ đi giá trị đó.

# 10. Độ phức tạp

Gọi `n = len(nums)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ |
|---|---|---:|---:|
| 1. Minimax | trạng thái `(i, ai đi)`, không nhớ | `O(φ^n)` (φ ≈ 1.618) | `O(n)` stack |
| 2. Negamax + nhớ | trạng thái `i`, `lru_cache` | `O(n)` | `O(n)` (+ stack đệ quy sâu `n`) |
| 3. DP lặp | mảng `f` từ cuối lên | `O(n)` | `O(n)` |
| 4. Hai biến | `f1`, `f2` | `O(n)` | `O(1)` |

Kiểm tra trên mảng `10^5` phần tử — cách 2 gặp giới hạn đệ quy, cách 4 chạy tức thì:

In [12]:
import random
import time

random.seed(4113)
big = [random.randint(1, 10**9) for _ in range(10**5)]

t0 = time.perf_counter()
r4 = sol.maxDifference(big)
print(f"Cách 4: {r4:,} ({time.perf_counter() - t0:.3f}s)")
print("Cách 3 cho cùng kết quả?", max_diff_bottom_up(big) == r4)

try:
    max_diff_memo(big)
except RecursionError as e:
    print("Cách 2: RecursionError -", str(e)[:45])

Cách 4: 42,041,364,935,573 (0.026s)


Cách 3 cho cùng kết quả? True
Cách 2: RecursionError - maximum recursion depth exceeded


# 11. Đối chiếu bốn cách trên dữ liệu ngẫu nhiên

In [13]:
random.seed(2026)
for _ in range(3000):
    nums = [random.randint(1, 12) for _ in range(random.randint(1, 12))]
    expected = max_diff_minimax(nums)
    assert max_diff_memo(nums) == expected, nums
    assert max_diff_bottom_up(nums) == expected, nums
    assert max_diff_o1(nums) == expected, nums
    assert sol.maxDifference(nums) == expected, nums

print("3000 bộ test ngẫu nhiên: 5 hàm cho cùng kết quả!")

3000 bộ test ngẫu nhiên: 5 hàm cho cùng kết quả!


# 12. Những lỗi dễ mắc

## Lỗi 1 — Luôn đổi lượt sau mỗi nước

Bỏ qua luật "không chia hết thì đi tiếp" → luôn dùng dấu trừ:

In [14]:
def wrong_always_switch(nums):
    n = len(nums)
    f = [0] * (n + 2)
    for i in range(n - 1, -1, -1):
        f[i] = nums[i] - f[i + 1]
        if i + 1 < n:
            f[i] = max(f[i], nums[i] + nums[i + 1] - f[i + 2])
    return f[0]


print("Sai :", wrong_always_switch([6, 1, 2, 1, 2]))
print("Đúng:", sol.maxDifference([6, 1, 2, 1, 2]))

Sai : 6
Đúng: 12


## Lỗi 2 — Đảo dấu: chia hết thì cộng, không chia hết thì trừ

Đọc nhầm luật, hoặc nhầm "đổi lượt" với "đi tiếp". Kiểm tra lại bằng ví dụ 1: lấy `9` (chia hết) phải **trao lượt** cho Bob → `9 - f[...]`.

In [15]:
def wrong_swapped_sign(nums):
    n = len(nums)
    f = [0] * (n + 2)
    for i in range(n - 1, -1, -1):
        x = nums[i]
        f[i] = x + f[i + 1] if x % 3 == 0 else x - f[i + 1]          # SAI: đảo dấu
        if i + 1 < n:
            s = x + nums[i + 1]
            f[i] = max(f[i], s + f[i + 2] if s % 3 == 0 else s - f[i + 2])
    return f[0]


print("Sai :", wrong_swapped_sign([3, 6, 4]))
print("Đúng:", sol.maxDifference([3, 6, 4]))

Sai : 13
Đúng: 5


## Lỗi 3 — Chơi tham lam

Một chiến lược "nghe hợp lý": *luôn chọn nước có tổng lấy được lớn hơn ngay lúc này*. Ta tìm một phản ví dụ bằng cách thử ngẫu nhiên:

In [16]:
def greedy_play(nums):
    # mỗi người luôn lấy nhóm có tổng lớn hơn (ưu tiên lấy 2 khi bằng nhau)
    i, n = 0, len(nums)
    alice_turn = True
    diff = 0
    while i < n:
        take = 2 if i + 1 < n else 1
        s = sum(nums[i:i + take])
        diff += s if alice_turn else -s
        if s % 3 == 0:
            alice_turn = not alice_turn
        i += take
    return diff


random.seed(1)
for _ in range(1000):
    arr = [random.randint(1, 9) for _ in range(random.randint(2, 6))]
    if greedy_play(arr) != sol.maxDifference(arr):
        print("Phản ví dụ:", arr, "| tham lam:", greedy_play(arr), "| tối ưu:", sol.maxDifference(arr))
        break
else:
    print("Không tìm thấy phản ví dụ trong 1000 lần thử")

# Một phản ví dụ dễ hiểu:
print("[1, 2, 5] | tham lam:", greedy_play([1, 2, 5]), "| tối ưu:", sol.maxDifference([1, 2, 5]))

Phản ví dụ: [4, 2, 6, 1] | tham lam: -1 | tối ưu: 13
[1, 2, 5] | tham lam: -2 | tối ưu: 8


Với `[1, 2, 5]`: tham lam lấy ngay `1 + 2 = 3` (chia hết) và trao lượt → Bob lấy `5` → hiệu `-2`. Tối ưu: Alice lấy `1` (không chia hết, giữ lượt) rồi lấy `2 + 5 = 7` → hiệu `8`.

Tham lam chỉ nhìn **một bước**, không tính tới việc mình có giữ được lượt hay không và đối thủ sẽ đứng ở vị trí nào. Trò chơi đòi hỏi nhìn **toàn bộ phần còn lại** — đó là việc của DP.

## Lỗi 4 — Quên kiểm tra còn ít nhất 2 phần tử

Viết `nums[i + 1]` khi `i = n - 1` sẽ báo `IndexError`. Luôn có điều kiện `if i + 1 < n`.

## Lỗi 5 — Lưu điểm của từng người trong trạng thái

Nếu trạng thái là `(i, điểm Alice, điểm Bob, ai đi)`, số trạng thái khổng lồ (điểm tới `10^14`). Mẹo hiệu điểm (mục 2.1) giúp trạng thái chỉ còn `i`.

## Lỗi 6 — Dùng đệ quy cho `n = 10^5`

Kể cả có `lru_cache`, độ sâu đệ quy vẫn là `n` → `RecursionError` (mục 10). Dùng DP lặp từ cuối mảng.

# 13. Mẹo nhận diện

1. **Hai người chơi tối ưu, tổng điểm cố định** (mọi phần tử đều thuộc về ai đó) → dùng **hiệu điểm** theo góc nhìn người đang đi.
2. Mỗi lượt lấy từ **một đầu** của mảng (ở đây là đầu trái) → trạng thái chỉ là **vị trí `i`** (hậu tố còn lại) → DP từ cuối mảng.
3. Luật "đi thêm lượt" / "mất lượt" → chỉ là **dấu `+` / `-`** trước giá trị của trạng thái tiếp theo.
4. Thấy `n` tới `10^5` → tránh đệ quy, dùng DP lặp; nếu chỉ phụ thuộc vài trạng thái kế tiếp → giữ vài biến.

Các bài cùng họ: *Stone Game III* (LeetCode 1406 — lấy 1–3 viên từ đầu, DP hiệu điểm y hệt), *Predict the Winner* (486), *Stone Game* (877), *Stone Game II* (1140).

# 14. Biến thể

## Biến thể 1 — Được lấy tới `m` phần tử mỗi lượt

Thay hai lựa chọn bằng vòng lặp `take = 1..m`. Duy trì tổng tiền tố của nhóm đang lấy.

In [17]:
def max_diff_take_up_to_m(nums: List[int], m: int) -> int:
    n = len(nums)
    f = [0] * (n + m + 1)
    for i in range(n - 1, -1, -1):
        best = None
        s = 0
        for take in range(1, m + 1):
            if i + take > n:
                break
            s += nums[i + take - 1]
            val = s - f[i + take] if s % 3 == 0 else s + f[i + take]
            best = val if best is None else max(best, val)
        f[i] = best
    return f[0]


print(max_diff_take_up_to_m([3, 3, 4, 5], 2))   # giống bài gốc: -3
print(max_diff_take_up_to_m([3, 3, 4, 5], 3))   # thêm lựa chọn lấy 3 phần tử

-3
15


## Biến thể 2 — Tính riêng điểm của Alice

Vì `Alice + Bob = T` và `Alice - Bob = D`, ta có:

```text
Alice = (T + D) / 2,     Bob = (T - D) / 2
```

In [18]:
def alice_and_bob_scores(nums: List[int]):
    T = sum(nums)
    D = sol.maxDifference(nums)
    return (T + D) // 2, (T - D) // 2


print(alice_and_bob_scores([3, 6, 4]))         # (9, 4)
print(alice_and_bob_scores([3, 3, 4, 5]))      # (6, 9)

(9, 4)
(6, 9)


# 15. Template tư duy

```python
# Trò chơi hai người, lấy từ đầu trái, tối đa hiệu điểm theo góc nhìn người đang đi
f = [0] * (n + EXTRA)                 # f[i] cho hậu tố nums[i:]
for i in range(n - 1, -1, -1):
    f[i] = max(
        gain(move) + (f[next] if keep_turn(move) else -f[next])
        for move in moves_from(i)
    )
answer = f[0]
```

# 16. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Lập bảng DP (như mục 6) cho:

```python
nums = [6, 3, 1, 2]
```

Đáp án là bao nhiêu? Mô tả ván đấu tối ưu (ai lấy gì). Kiểm tra bằng cell dưới.

In [19]:
# Kiểm tra Bài 1
trace_bottom_up([6, 3, 1, 2]);

nums = [6, 3, 1, 2]
i = 3: lấy 2 (đi tiếp): 2 + 0 = 2 -> f[3] = 2
i = 2: lấy 1 (đi tiếp): 1 + 2 = 3 | lấy 1+2=3 (đổi lượt): 3 - 0 = 3 -> f[2] = 3
i = 1: lấy 3 (chia hết → đổi lượt): 3 - 3 = 0 | lấy 3+1=4 (đi tiếp): 4 + 2 = 6 -> f[1] = 6
i = 0: lấy 6 (chia hết → đổi lượt): 6 - 6 = 0 | lấy 6+3=9 (đổi lượt): 9 - 3 = 6 -> f[0] = 6
Đáp án f[0] = 6


## Bài 2 — Dự đoán output

```python
print(max_diff_bottom_up([5]))
print(max_diff_bottom_up([2, 1]))
print(max_diff_bottom_up([3, 3, 3]))
```

Gợi ý cho dòng cuối: lấy từng số `3` thì lượt đổi liên tục; lấy hai số `3` (tổng `6`) cũng đổi lượt.

In [20]:
print(max_diff_bottom_up([5]))
print(max_diff_bottom_up([2, 1]))
print(max_diff_bottom_up([3, 3, 3]))

5
3
3


## Bài 3 — Hoàn thiện DP lặp

In [21]:
def exercise_3_bottom_up(nums: List[int]) -> int:
    n = len(nums)
    f = [0] * (n + 2)
    for i in range(n - 1, -1, -1):
        # TODO 1: giá trị khi lấy 1 phần tử (chú ý dấu theo luật chia hết cho 3)
        # TODO 2: nếu còn >= 2 phần tử, giá trị khi lấy 2 phần tử
        # TODO 3: f[i] = max của các lựa chọn
        pass
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_bottom_up([3, 6, 4]) == 5
# assert exercise_3_bottom_up([3, 3, 4, 5]) == -3

## Bài 4 — Ai thắng?

Viết hàm trả về `"Alice"`, `"Bob"` hoặc `"Hòa"` tùy theo hiệu điểm khi cả hai chơi tối ưu.

```python
winner([3, 6, 4])      -> "Alice"
winner([3, 3, 4, 5])   -> "Bob"
```

In [22]:
def exercise_4_winner(nums: List[int]) -> str:
    # TODO
    raise NotImplementedError


# assert exercise_4_winner([3, 6, 4]) == "Alice"
# assert exercise_4_winner([3, 3, 4, 5]) == "Bob"

## Bài 5 — Biến thể: đảo luật

Đổi luật: tổng **chia hết** cho 3 thì **cùng người đi tiếp**; **không chia hết** thì **đổi lượt**. Tính hiệu `Alice - Bob`.

```python
max_diff_inverted([3, 6, 4])   -> ?   (tự tính bằng tay trước, rồi kiểm tra)
```

Yêu cầu: viết DP lặp, kiểm tra bằng một hàm minimax tự viết trên dữ liệu ngẫu nhiên nhỏ.

In [23]:
def exercise_5_max_diff_inverted(nums: List[int]) -> int:
    # TODO
    raise NotImplementedError


# print(exercise_5_max_diff_inverted([3, 6, 4]))

## Bài 6 — Mini challenge: tái hiện ván đấu tối ưu

Trả về danh sách các nước đi theo lối chơi tối ưu, mỗi nước là `(người chơi, các phần tử đã lấy)`.

```python
replay([6, 1, 2, 1, 2])
-> [("Alice", [6, 1]), ("Alice", [2]), ("Alice", [1, 2])]
```

Kiểm tra: hiệu điểm tính từ danh sách nước đi phải bằng `maxDifference`.

Gợi ý: tính bảng `f` trước, rồi đi từ `i = 0`: ở mỗi bước chọn nước đạt đúng giá trị `f[i]`.

In [24]:
def exercise_6_replay(nums: List[int]):
    # TODO
    raise NotImplementedError


# moves = exercise_6_replay([6, 1, 2, 1, 2])
# assert moves == [("Alice", [6, 1]), ("Alice", [2]), ("Alice", [1, 2])]

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: lấy `x = nums[i]`: chia hết → `x - f[i+1]`, không → `x + f[i+1]`.
- Mức 2: lấy `s = nums[i] + nums[i+1]` (khi `i + 1 < n`): chia hết → `s - f[i+2]`, không → `s + f[i+2]`.
- Mức 3: `f[i] = max(...)`, cuối cùng `return f[0]`.

**Bài 4**

- Mức 1: dùng `sol.maxDifference(nums)`.
- Mức 2: dương → Alice, âm → Bob, bằng 0 → Hòa.
- Mức 3: ba nhánh `if / elif / else`.

**Bài 5**

- Mức 1: chỉ đổi điều kiện chọn dấu.
- Mức 2: chia hết → `+ f[next]` (giữ lượt), không chia hết → `- f[next]` (đổi lượt).
- Mức 3: viết minimax với trạng thái `(i, alice_turn)` để đối chiếu.

**Bài 6**

- Mức 1: xây mảng `f` như cách 3.
- Mức 2: tại `i`, tính giá trị của nước "lấy 1" và "lấy 2"; chọn nước có giá trị bằng `f[i]`.
- Mức 3: ghi lại `(người, nhóm)`; nếu tổng nhóm chia hết cho 3 thì đổi người; nhảy `i` qua nhóm vừa lấy.

# 17. Đáp án tham khảo

Bạn nên tự làm trước khi mở phần này.

## Đáp án Bài 1

`nums = [6, 3, 1, 2]`:

| i | nums[i] | lấy 1 | lấy 2 | f[i] |
|---:|---:|---|---|---:|
| 4 | — | | | 0 |
| 3 | 2 | `2` → `2 + 0 = 2` | — | 2 |
| 2 | 1 | `1` → `1 + 2 = 3` | `1+2 = 3` chia hết → `3 - 0 = 3` | 3 |
| 1 | 3 | `3` chia hết → `3 - 3 = 0` | `3+1 = 4` → `4 + 2 = 6` | 6 |
| 0 | 6 | `6` chia hết → `6 - 6 = 0` | `6+3 = 9` chia hết → `9 - 3 = 6` | **6** |

Đáp án `6`. Ván đấu: Alice lấy `6 + 3 = 9` (chia hết → Bob). Bob đứng trước `[1, 2]` với `f[2] = 3`: Bob lấy được cả `3` điểm còn lại. Alice `9`, Bob `3` → hiệu `6`.

(Nếu Alice chỉ lấy `6`, lượt sang Bob ở vị trí `f[1] = 6` — Bob hơn Alice 6 điểm ở phần còn lại, tổng hiệu chỉ còn `0`.)

## Đáp án Bài 2

```text
5
3
3
```

- `[5]`: Alice lấy `5` → `5`.
- `[2, 1]`: Alice lấy `2` (không chia hết → đi tiếp) rồi lấy `1`; hoặc lấy `2 + 1 = 3` cùng lúc (chia hết nhưng mảng đã hết). Cả hai cách đều được `3`.
- `[3, 3, 3]`: Alice lấy `3 + 3 = 6` (→ Bob), Bob lấy `3` cuối → `6 - 3 = 3`. Nếu Alice chỉ lấy một số `3`, Bob đứng trước `[3, 3]` và lấy cả hai (`6`) → hiệu `3 - 6 = -3`, tệ hơn.

In [25]:
# Đáp án Bài 3
def answer_3_bottom_up(nums: List[int]) -> int:
    n = len(nums)
    f = [0] * (n + 2)
    for i in range(n - 1, -1, -1):
        x = nums[i]
        f[i] = x - f[i + 1] if x % 3 == 0 else x + f[i + 1]
        if i + 1 < n:
            s = x + nums[i + 1]
            f[i] = max(f[i], s - f[i + 2] if s % 3 == 0 else s + f[i + 2])
    return f[0]


assert answer_3_bottom_up([3, 6, 4]) == 5
assert answer_3_bottom_up([3, 3, 4, 5]) == -3
assert answer_3_bottom_up([6, 3, 1, 2]) == 6
print("Bài 3: PASS")

Bài 3: PASS


In [26]:
# Đáp án Bài 4
def answer_4_winner(nums: List[int]) -> str:
    d = sol.maxDifference(nums)
    if d > 0:
        return "Alice"
    if d < 0:
        return "Bob"
    return "Hòa"


assert answer_4_winner([3, 6, 4]) == "Alice"
assert answer_4_winner([3, 3, 4, 5]) == "Bob"
print("Bài 4: PASS |", answer_4_winner([3, 3]), "với [3, 3]")

Bài 4: PASS | Alice với [3, 3]


In [27]:
# Đáp án Bài 5
def answer_5_max_diff_inverted(nums: List[int]) -> int:
    n = len(nums)
    f = [0] * (n + 2)
    for i in range(n - 1, -1, -1):
        x = nums[i]
        f[i] = x + f[i + 1] if x % 3 == 0 else x - f[i + 1]       # chia hết -> giữ lượt
        if i + 1 < n:
            s = x + nums[i + 1]
            f[i] = max(f[i], s + f[i + 2] if s % 3 == 0 else s - f[i + 2])
    return f[0]


def minimax_inverted(nums: List[int]) -> int:
    n = len(nums)

    def go(i, alice_turn):
        if i == n:
            return 0
        res = []
        for take in (1, 2):
            if i + take > n:
                break
            s = sum(nums[i:i + take])
            nxt = alice_turn if s % 3 == 0 else (not alice_turn)
            r = go(i + take, nxt)
            res.append(r + s if alice_turn else r - s)
        return max(res) if alice_turn else min(res)

    return go(0, True)


print("[3, 6, 4] ->", answer_5_max_diff_inverted([3, 6, 4]))
random.seed(5)
for _ in range(1000):
    arr = [random.randint(1, 12) for _ in range(random.randint(1, 10))]
    assert answer_5_max_diff_inverted(arr) == minimax_inverted(arr), arr
print("Bài 5: PASS (khớp minimax trên 1000 test ngẫu nhiên)")

[3, 6, 4] -> 13
Bài 5: PASS (khớp minimax trên 1000 test ngẫu nhiên)


In [28]:
# Đáp án Bài 6
def answer_6_replay(nums: List[int]):
    n = len(nums)
    f = [0] * (n + 2)
    for i in range(n - 1, -1, -1):
        x = nums[i]
        f[i] = x - f[i + 1] if x % 3 == 0 else x + f[i + 1]
        if i + 1 < n:
            s = x + nums[i + 1]
            f[i] = max(f[i], s - f[i + 2] if s % 3 == 0 else s + f[i + 2])

    moves = []
    player = "Alice"
    i = 0
    while i < n:
        for take in (2, 1):                      # thử lấy 2 trước để ví dụ khớp đề
            if i + take > n:
                continue
            s = sum(nums[i:i + take])
            val = s - f[i + take] if s % 3 == 0 else s + f[i + take]
            if val == f[i]:
                moves.append((player, nums[i:i + take]))
                if s % 3 == 0:
                    player = "Bob" if player == "Alice" else "Alice"
                i += take
                break
    return moves


def diff_from_moves(moves):
    return sum(sum(g) if p == "Alice" else -sum(g) for p, g in moves)


moves = answer_6_replay([6, 1, 2, 1, 2])
print(moves)
assert moves == [("Alice", [6, 1]), ("Alice", [2]), ("Alice", [1, 2])]
print(answer_6_replay([3, 3, 4, 5]))
random.seed(6)
for _ in range(1000):
    arr = [random.randint(1, 12) for _ in range(random.randint(1, 10))]
    assert diff_from_moves(answer_6_replay(arr)) == sol.maxDifference(arr), arr
print("Bài 6: PASS (ván tái hiện luôn cho đúng hiệu điểm tối ưu)")

[('Alice', [6, 1]), ('Alice', [2]), ('Alice', [1, 2])]
[('Alice', [3, 3]), ('Bob', [4, 5])]
Bài 6: PASS (ván tái hiện luôn cho đúng hiệu điểm tối ưu)


# 18. Tổng kết

Bài **Game of Divisible Sums** là bài nhập môn tuyệt vời cho DP trò chơi:

1. **Tổng điểm cố định** → tối đa điểm mình ⇔ tối đa **hiệu điểm**.
2. **Negamax:** `f[i]` theo góc nhìn người đang đi; trạng thái chỉ còn `i`.
3. Luật lượt đi → **dấu**: giữ lượt thì `+ f[next]`, trao lượt thì `- f[next]`.
4. DP từ cuối mảng, rồi rút gọn còn **hai biến**.

## Công thức nhớ nhanh

```text
f[i] = max( x       - f[i+1]  nếu x % 3 == 0       else x       + f[i+1],
            (x + y) - f[i+2]  nếu (x+y) % 3 == 0   else (x + y) + f[i+2] )
answer = f[0]
```

## Phiên bản nên nhớ

```python
class Solution:
    def maxDifference(self, nums: List[int]) -> int:
        n = len(nums)
        f1 = f2 = 0
        for i in range(n - 1, -1, -1):
            x = nums[i]
            cur = x - f1 if x % 3 == 0 else x + f1
            if i + 1 < n:
                s = x + nums[i + 1]
                cur = max(cur, s - f2 if s % 3 == 0 else s + f2)
            f2, f1 = f1, cur
        return f1
```

## Độ phức tạp

```text
Time:  O(n)
Space: O(1)
```

# 19. Checklist trước khi submit

- [ ] Hiểu đúng luật: tổng **chia hết** cho 3 → **đổi lượt**; không chia hết → **đi tiếp**?
- [ ] Dấu đúng: đổi lượt → `s - f[next]`; đi tiếp → `s + f[next]`?
- [ ] Chỉ xét lấy 2 phần tử khi `i + 1 < n`?
- [ ] Trả về **hiệu** `Alice - Bob` (không phải điểm của Alice)?
- [ ] Không dùng đệ quy cho `n = 10^5` (kể cả có `lru_cache`)?
- [ ] Khi rút gọn bộ nhớ: dịch cửa sổ đúng thứ tự `f2, f1 = f1, cur`, trả về `f1` sau vòng lặp?
- [ ] Đã test: một phần tử, không nước nào chia hết, toàn bội của 3, số lớn?